In [1]:
"""
Station-Level Regression: Did Nearby Accidents Predict Union Branches?
=======================================================================

Estimating equation:
    has_union_i = α + β × has_accident_i + X_i'γ + ε_i

where i indexes stations, and the key parameter of interest is β.

This script:
1. Loads the station-level merged dataset (output of spatial analysis pipeline)
2. Runs the baseline LPM and Logit regressions
3. Performs robustness checks across multiple buffer radii (1km, 2km, 5km, 10km, 15km)
4. Adds spatial controls (latitude/longitude polynomials, region FE)
5. Exports a formatted regression table

Requires: statsmodels, geopandas (for multi-radius), pandas, numpy, scipy
If statsmodels unavailable: falls back to scipy logit + sklearn

Author: Keitaro Ninomiya
"""

import pandas as pd
import numpy as np
from pathlib import Path
import os
import warnings
warnings.filterwarnings("ignore")


# =============================================================================
# CONFIGURATION
# =============================================================================

base_path = r"C:\Users\Keitaro Ninomiya\Box\Research Notes (keitaro2@illinois.edu)\RailwayUnions\Processed_Data"

# Input: station-level dataset from the spatial analysis pipeline
STATION_DATA = os.path.join(base_path, "station_level_merged_data.csv")

# Input: raw data for re-computing at different radii
STATIONS_SHP = os.path.join(base_path, r"5. 1861 England, Wales and Scotland rail stations\1861EngWalesScotRail_Stations.shp")
ACCIDENT_CACHE = os.path.join(base_path, "cache_accident_geocoding.csv")
UNION_CACHE = os.path.join(base_path, "cache_union_geocoding.csv")
ACCIDENTS_CSV = os.path.join(base_path, "detailed_accidents_data.csv")
UNIONS_CSV = os.path.join(base_path, r"ASRS\BalanceSheets\1875\Results\georeferenced_railway_results.csv")

# Output
REGRESSION_TABLE = os.path.join(base_path, "regression_results.csv")
REGRESSION_TEX = os.path.join(base_path, "regression_results.tex")

# Buffer radii to test (meters)
RADII = [1000, 2000, 5000, 10000, 15000]

# Optional: filter accidents by year range
YEAR_MIN = None  # e.g., 1861
YEAR_MAX = None  # e.g., 1875


# =============================================================================
# MULTI-RADIUS SPATIAL JOIN
# =============================================================================

def compute_multi_radius_dataset(radii: list) -> pd.DataFrame:
    """
    Recompute has_accident and has_union for multiple buffer radii.
    Returns station-level DataFrame with columns like:
        has_accident_1km, has_accident_2km, ..., has_union_1km, ...
    """
    import geopandas as gpd

    print("Loading spatial data for multi-radius analysis...")
    stations = gpd.read_file(STATIONS_SHP).to_crs(epsg=27700)

    # --- Load and prepare accidents ---
    acc_raw = pd.read_csv(ACCIDENTS_CSV)
    if YEAR_MIN is not None:
        acc_raw = acc_raw[acc_raw["year"] >= YEAR_MIN]
    if YEAR_MAX is not None:
        acc_raw = acc_raw[acc_raw["year"] <= YEAR_MAX]

    acc_geo = pd.read_csv(ACCIDENT_CACHE)
    acc_raw["_join"] = acc_raw["location"].astype(str).str.lower().str.strip()
    acc_geo["_join"] = acc_geo["location"].astype(str).str.lower().str.strip()
    acc_geo = acc_geo.drop_duplicates(subset=["_join"])

    acc_merged = acc_raw.merge(acc_geo[["_join", "latitude", "longitude"]], on="_join", how="left")
    acc_merged = acc_merged.dropna(subset=["latitude", "longitude"])
    acc_gdf = gpd.GeoDataFrame(
        acc_merged,
        geometry=gpd.points_from_xy(acc_merged["longitude"], acc_merged["latitude"]),
        crs="EPSG:4326"
    ).to_crs(epsg=27700)

    # --- Load and prepare union branches ---
    br_raw = pd.read_csv(UNIONS_CSV).dropna(subset=["cleaned_loc"])
    br_geo = pd.read_csv(UNION_CACHE)
    br_raw["_join"] = br_raw["cleaned_loc"].astype(str).str.lower().str.strip()
    # The union cache may use corrected_loc names; handle both
    if "corrected_loc" in br_raw.columns:
        br_raw["_join"] = br_raw["corrected_loc"].astype(str).str.lower().str.strip()
    br_geo["_join"] = br_geo["location"].astype(str).str.lower().str.strip()
    br_geo = br_geo.drop_duplicates(subset=["_join"])

    br_merged = br_raw.merge(br_geo[["_join", "latitude", "longitude"]], on="_join", how="left")
    br_merged = br_merged.dropna(subset=["latitude", "longitude"])
    br_gdf = gpd.GeoDataFrame(
        br_merged,
        geometry=gpd.points_from_xy(br_merged["longitude"], br_merged["latitude"]),
        crs="EPSG:4326"
    ).to_crs(epsg=27700)

    print(f"  Stations: {len(stations)}, Accidents: {len(acc_gdf)}, Branches: {len(br_gdf)}")

    # --- Compute counts at each radius ---
    result = stations[["Id"]].copy()
    result["easting"] = stations.geometry.x
    result["northing"] = stations.geometry.y
    stations_wgs84 = stations.to_crs("EPSG:4326")
    result["latitude"] = stations_wgs84.geometry.y
    result["longitude"] = stations_wgs84.geometry.x

    for radius in radii:
        label = f"{radius // 1000}km" if radius >= 1000 else f"{radius}m"
        print(f"  Computing buffer radius = {label}...")

        buffers = stations[["Id", "geometry"]].copy()
        buffers["geometry"] = buffers.geometry.buffer(radius)

        # Accidents
        acc_join = gpd.sjoin(
            acc_gdf[["geometry"]].reset_index(drop=True),
            buffers, how="inner", predicate="intersects"
        )
        acc_counts = acc_join.groupby("Id").size().reset_index(name=f"accident_count_{label}")
        result = result.merge(acc_counts, on="Id", how="left")
        result[f"accident_count_{label}"] = result[f"accident_count_{label}"].fillna(0).astype(int)
        result[f"has_accident_{label}"] = (result[f"accident_count_{label}"] > 0).astype(int)

        # Unions
        br_join = gpd.sjoin(
            br_gdf[["geometry"]].reset_index(drop=True),
            buffers, how="inner", predicate="intersects"
        )
        br_counts = br_join.groupby("Id").size().reset_index(name=f"union_count_{label}")
        result = result.merge(br_counts, on="Id", how="left")
        result[f"union_count_{label}"] = result[f"union_count_{label}"].fillna(0).astype(int)
        result[f"has_union_{label}"] = (result[f"union_count_{label}"] > 0).astype(int)

        n_acc = result[f"has_accident_{label}"].sum()
        n_uni = result[f"has_union_{label}"].sum()
        print(f"    Stations with accident: {n_acc} ({n_acc/len(result):.1%})")
        print(f"    Stations with union:    {n_uni} ({n_uni/len(result):.1%})")

    return result


# =============================================================================
# REGRESSION FUNCTIONS
# =============================================================================

def run_regressions_statsmodels(df: pd.DataFrame, radii_labels: list) -> pd.DataFrame:
    """
    Run LPM and Logit regressions using statsmodels.

    Models:
    1. LPM (OLS): has_union = α + β·has_accident + ε
    2. LPM + spatial controls: + lat + lon + lat² + lon²
    3. Logit: same specifications
    4. Robustness across buffer radii

    Returns a DataFrame of regression results.
    """
    import statsmodels.api as sm
    from statsmodels.discrete.discrete_model import Logit

    results_rows = []

    for label in radii_labels:
        y_col = f"has_union_{label}"
        x_col = f"has_accident_{label}"
        acc_count_col = f"accident_count_{label}"

        if y_col not in df.columns:
            continue

        y = df[y_col].values
        n = len(y)

        # Skip if no variation
        if y.sum() == 0 or y.sum() == n:
            print(f"  SKIP {label}: no variation in outcome (mean={y.mean():.4f})")
            results_rows.append({
                "radius": label, "model": "—", "beta": np.nan,
                "se": np.nan, "p_value": np.nan, "n_obs": n,
                "y_mean": y.mean(), "x_mean": df[x_col].mean(),
                "note": "No variation in outcome"
            })
            continue

        # ---------------------------------------------------------------
        # Model 1: LPM (OLS), binary treatment
        # ---------------------------------------------------------------
        X1 = sm.add_constant(df[[x_col]])
        lpm1 = sm.OLS(y, X1).fit(cov_type="HC1")

        results_rows.append({
            "radius": label,
            "model": "LPM (binary)",
            "beta": lpm1.params[x_col],
            "se": lpm1.bse[x_col],
            "p_value": lpm1.pvalues[x_col],
            "ci_lower": lpm1.conf_int().loc[x_col, 0],
            "ci_upper": lpm1.conf_int().loc[x_col, 1],
            "r_squared": lpm1.rsquared,
            "n_obs": int(lpm1.nobs),
            "y_mean": y.mean(),
            "x_mean": df[x_col].mean(),
            "note": "HC1 robust SEs"
        })

        # ---------------------------------------------------------------
        # Model 2: LPM, count treatment (intensive margin)
        # ---------------------------------------------------------------
        X2 = sm.add_constant(df[[acc_count_col]])
        lpm2 = sm.OLS(y, X2).fit(cov_type="HC1")

        results_rows.append({
            "radius": label,
            "model": "LPM (count)",
            "beta": lpm2.params[acc_count_col],
            "se": lpm2.bse[acc_count_col],
            "p_value": lpm2.pvalues[acc_count_col],
            "ci_lower": lpm2.conf_int().loc[acc_count_col, 0],
            "ci_upper": lpm2.conf_int().loc[acc_count_col, 1],
            "r_squared": lpm2.rsquared,
            "n_obs": int(lpm2.nobs),
            "y_mean": y.mean(),
            "x_mean": df[acc_count_col].mean(),
            "note": "HC1 robust SEs"
        })

        # ---------------------------------------------------------------
        # Model 3: LPM + spatial controls
        # ---------------------------------------------------------------
        spatial_controls = pd.DataFrame({
            "lat": df["latitude"],
            "lon": df["longitude"],
            "lat2": df["latitude"] ** 2,
            "lon2": df["longitude"] ** 2,
            "lat_lon": df["latitude"] * df["longitude"],
        })
        X3 = sm.add_constant(pd.concat([df[[x_col]], spatial_controls], axis=1))
        lpm3 = sm.OLS(y, X3).fit(cov_type="HC1")

        results_rows.append({
            "radius": label,
            "model": "LPM + spatial",
            "beta": lpm3.params[x_col],
            "se": lpm3.bse[x_col],
            "p_value": lpm3.pvalues[x_col],
            "ci_lower": lpm3.conf_int().loc[x_col, 0],
            "ci_upper": lpm3.conf_int().loc[x_col, 1],
            "r_squared": lpm3.rsquared,
            "n_obs": int(lpm3.nobs),
            "y_mean": y.mean(),
            "x_mean": df[x_col].mean(),
            "note": "HC1 robust SEs; controls: lat, lon, lat², lon², lat×lon"
        })

        # ---------------------------------------------------------------
        # Model 4: Logit (binary treatment)
        # ---------------------------------------------------------------
        try:
            X4 = sm.add_constant(df[[x_col]])
            logit_model = Logit(y, X4).fit(disp=0, maxiter=100)

            # Marginal effect at the mean
            mfx = logit_model.get_margeff(at="mean")

            results_rows.append({
                "radius": label,
                "model": "Logit (ME at mean)",
                "beta": mfx.margeff[0],
                "se": mfx.margeff_se[0],
                "p_value": mfx.pvalues[0],
                "ci_lower": mfx.conf_int()[0, 0],
                "ci_upper": mfx.conf_int()[0, 1],
                "r_squared": logit_model.prsquared,  # McFadden's pseudo R²
                "n_obs": int(logit_model.nobs),
                "y_mean": y.mean(),
                "x_mean": df[x_col].mean(),
                "note": "Marginal effects at mean; McFadden pseudo-R²"
            })
        except Exception as e:
            results_rows.append({
                "radius": label,
                "model": "Logit",
                "beta": np.nan, "se": np.nan, "p_value": np.nan,
                "n_obs": n, "y_mean": y.mean(), "x_mean": df[x_col].mean(),
                "note": f"Logit failed: {str(e)[:60]}"
            })

        # ---------------------------------------------------------------
        # Model 5: Logit + spatial controls
        # ---------------------------------------------------------------
        try:
            X5 = sm.add_constant(pd.concat([df[[x_col]], spatial_controls], axis=1))
            logit5 = Logit(y, X5).fit(disp=0, maxiter=100)
            mfx5 = logit5.get_margeff(at="mean")

            results_rows.append({
                "radius": label,
                "model": "Logit + spatial (ME)",
                "beta": mfx5.margeff[0],
                "se": mfx5.margeff_se[0],
                "p_value": mfx5.pvalues[0],
                "ci_lower": mfx5.conf_int()[0, 0],
                "ci_upper": mfx5.conf_int()[0, 1],
                "r_squared": logit5.prsquared,
                "n_obs": int(logit5.nobs),
                "y_mean": y.mean(),
                "x_mean": df[x_col].mean(),
                "note": "ME at mean; spatial controls"
            })
        except Exception as e:
            pass  # Skip quietly

    return pd.DataFrame(results_rows)


def run_regressions_fallback(df: pd.DataFrame, radii_labels: list) -> pd.DataFrame:
    """
    Fallback regression using scipy + numpy when statsmodels unavailable.
    Implements OLS manually with HC1 standard errors.
    """
    from scipy import stats

    results_rows = []

    for label in radii_labels:
        y_col = f"has_union_{label}"
        x_col = f"has_accident_{label}"

        if y_col not in df.columns:
            continue

        y = df[y_col].values.astype(float)
        x = df[x_col].values.astype(float)
        n = len(y)

        if y.sum() == 0 or y.sum() == n:
            results_rows.append({
                "radius": label, "model": "—", "beta": np.nan,
                "se": np.nan, "p_value": np.nan, "n_obs": n,
                "y_mean": y.mean(), "x_mean": x.mean(),
                "note": "No variation in outcome"
            })
            continue

        # --- OLS by hand with HC1 ---
        X = np.column_stack([np.ones(n), x])
        beta_hat = np.linalg.lstsq(X, y, rcond=None)[0]
        residuals = y - X @ beta_hat
        r_sq = 1 - np.sum(residuals**2) / np.sum((y - y.mean())**2)

        # HC1 robust variance
        leverage = X @ np.linalg.inv(X.T @ X) @ X.T
        hc1_factor = n / (n - X.shape[1])
        meat = X.T @ np.diag(residuals**2 * hc1_factor) @ X
        bread = np.linalg.inv(X.T @ X)
        V_hc1 = bread @ meat @ bread
        se_hc1 = np.sqrt(np.diag(V_hc1))

        t_stat = beta_hat[1] / se_hc1[1]
        p_value = 2 * stats.t.sf(abs(t_stat), df=n - 2)

        results_rows.append({
            "radius": label,
            "model": "LPM (binary)",
            "beta": beta_hat[1],
            "se": se_hc1[1],
            "p_value": p_value,
            "ci_lower": beta_hat[1] - 1.96 * se_hc1[1],
            "ci_upper": beta_hat[1] + 1.96 * se_hc1[1],
            "r_squared": r_sq,
            "n_obs": n,
            "y_mean": y.mean(),
            "x_mean": x.mean(),
            "note": "HC1 robust SEs (manual OLS)"
        })

        # --- LPM + spatial controls ---
        lat = df["latitude"].values
        lon = df["longitude"].values
        X_full = np.column_stack([
            np.ones(n), x, lat, lon, lat**2, lon**2, lat * lon
        ])
        try:
            beta_full = np.linalg.lstsq(X_full, y, rcond=None)[0]
            resid_full = y - X_full @ beta_full
            r_sq_full = 1 - np.sum(resid_full**2) / np.sum((y - y.mean())**2)

            k = X_full.shape[1]
            hc1_f = n / (n - k)
            meat_f = X_full.T @ np.diag(resid_full**2 * hc1_f) @ X_full
            bread_f = np.linalg.inv(X_full.T @ X_full)
            V_f = bread_f @ meat_f @ bread_f
            se_f = np.sqrt(np.diag(V_f))

            t_f = beta_full[1] / se_f[1]
            p_f = 2 * stats.t.sf(abs(t_f), df=n - k)

            results_rows.append({
                "radius": label,
                "model": "LPM + spatial",
                "beta": beta_full[1],
                "se": se_f[1],
                "p_value": p_f,
                "ci_lower": beta_full[1] - 1.96 * se_f[1],
                "ci_upper": beta_full[1] + 1.96 * se_f[1],
                "r_squared": r_sq_full,
                "n_obs": n,
                "y_mean": y.mean(),
                "x_mean": x.mean(),
                "note": "HC1 robust SEs; controls: lat, lon, lat², lon², lat×lon"
            })
        except np.linalg.LinAlgError:
            pass

    return pd.DataFrame(results_rows)


# =============================================================================
# DESCRIPTIVE STATISTICS
# =============================================================================

def print_descriptives(df: pd.DataFrame, radii_labels: list):
    """Print descriptive statistics for the regression sample."""
    print("\n" + "=" * 80)
    print("DESCRIPTIVE STATISTICS")
    print("=" * 80)
    print(f"  Total stations: {len(df)}")

    rows = []
    for label in radii_labels:
        y_col = f"has_union_{label}"
        x_col = f"has_accident_{label}"
        ac_col = f"accident_count_{label}"
        uc_col = f"union_count_{label}"

        if y_col not in df.columns:
            continue

        row = {
            "Radius": label,
            "Stations w/ accident": f"{df[x_col].sum()} ({df[x_col].mean():.1%})",
            "Stations w/ union": f"{df[y_col].sum()} ({df[y_col].mean():.1%})",
            "Mean acc. count": f"{df[ac_col].mean():.2f}",
            "Mean union count": f"{df[uc_col].mean():.2f}",
        }

        # Conditional means
        with_acc = df[df[x_col] == 1][y_col].mean() if df[x_col].sum() > 0 else np.nan
        without_acc = df[df[x_col] == 0][y_col].mean() if (df[x_col] == 0).sum() > 0 else np.nan
        row["P(union | accident)"] = f"{with_acc:.3f}" if not np.isnan(with_acc) else "—"
        row["P(union | no accident)"] = f"{without_acc:.3f}" if not np.isnan(without_acc) else "—"
        row["Raw difference"] = f"{with_acc - without_acc:.3f}" if not (np.isnan(with_acc) or np.isnan(without_acc)) else "—"

        rows.append(row)

    desc_df = pd.DataFrame(rows)
    print(desc_df.to_string(index=False))
    return desc_df


# =============================================================================
# LaTeX TABLE EXPORT
# =============================================================================

def export_latex_table(results: pd.DataFrame, output_path: str):
    """Export regression results as a LaTeX table."""

    # Filter to key specifications
    key_results = results[results["model"].isin([
        "LPM (binary)", "LPM + spatial", "Logit (ME at mean)"
    ])].copy()

    if len(key_results) == 0:
        print("  No results to export to LaTeX.")
        return

    def format_coef(row):
        if pd.isna(row["beta"]):
            return "—"
        stars = ""
        if row["p_value"] < 0.01:
            stars = "***"
        elif row["p_value"] < 0.05:
            stars = "**"
        elif row["p_value"] < 0.10:
            stars = "*"
        return f"{row['beta']:.4f}{stars}"

    def format_se(row):
        if pd.isna(row["se"]):
            return ""
        return f"({row['se']:.4f})"

    lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{Did Nearby Accidents Predict Union Branch Establishment?}",
        r"\label{tab:accident_union_regression}",
        r"\small",
        r"\begin{tabular}{l" + "c" * len(key_results["radius"].unique()) + "}",
        r"\hline\hline",
    ]

    # Header row with radii
    radii = key_results["radius"].unique()
    header = " & ".join([f"({i+1})" for i in range(len(radii))]) 
    lines.append(r" & " + header + r" \\")
    radius_row = " & ".join(radii)
    lines.append(r"Radius: & " + radius_row + r" \\")
    lines.append(r"\hline")

    # For each model type, show coefficients across radii
    for model in ["LPM (binary)", "LPM + spatial", "Logit (ME at mean)"]:
        subset = key_results[key_results["model"] == model]
        if len(subset) == 0:
            continue

        lines.append(r"\multicolumn{" + str(len(radii) + 1) + r"}{l}{\textit{" + model + r"}} \\")

        coefs = []
        ses = []
        for r in radii:
            row = subset[subset["radius"] == r]
            if len(row) > 0:
                coefs.append(format_coef(row.iloc[0]))
                ses.append(format_se(row.iloc[0]))
            else:
                coefs.append("—")
                ses.append("")

        lines.append(r"has\_accident & " + " & ".join(coefs) + r" \\")
        lines.append(r" & " + " & ".join(ses) + r" \\")
        lines.append(r"\\")

    # Footer
    lines.append(r"\hline")
    n_row = " & ".join([str(int(key_results[key_results["radius"] == r].iloc[0]["n_obs"])) 
                         if len(key_results[key_results["radius"] == r]) > 0 else "—" 
                         for r in radii])
    lines.append(r"N & " + n_row + r" \\")
    lines.append(r"\hline\hline")
    lines.append(r"\multicolumn{" + str(len(radii) + 1) + r"}{p{0.9\textwidth}}{")
    lines.append(r"\footnotesize Notes: Dependent variable is an indicator for having $\geq 1$ union branch within the specified radius. ")
    lines.append(r"Robust (HC1) standard errors in parentheses. ")
    lines.append(r"Spatial controls: latitude, longitude, and their squares and interaction. ")
    lines.append(r"$^{***}p<0.01$, $^{**}p<0.05$, $^{*}p<0.10$.}")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")

    with open(output_path, "w") as f:
        f.write("\n".join(lines))

    print(f"  LaTeX table saved: {output_path}")


# =============================================================================
# MAIN
# =============================================================================

def main():
    print("=" * 80)
    print("STATION-LEVEL REGRESSION: ACCIDENTS → UNION BRANCHES")
    print("=" * 80)

    # ------------------------------------------------------------------
    # Step 1: Build multi-radius dataset
    # ------------------------------------------------------------------
    try:
        df = compute_multi_radius_dataset(RADII)
        multi_radius_path = os.path.join(base_path, "station_multi_radius.csv")
        df.to_csv(multi_radius_path, index=False)
        print(f"\n  Multi-radius dataset saved: {multi_radius_path}")
    except ImportError:
        # geopandas not available — try loading pre-computed data
        print("  geopandas not available. Looking for pre-computed station data...")
        if Path(STATION_DATA).exists():
            df = pd.read_csv(STATION_DATA)
            print(f"  Loaded {len(df)} stations from {STATION_DATA}")
            # Rename 5km columns to match multi-radius format
            renames = {
                "has_accident": "has_accident_5km",
                "has_union": "has_union_5km",
                "accident_count": "accident_count_5km",
                "union_count": "union_count_5km",
            }
            df = df.rename(columns=renames)
            RADII.clear()
            RADII.append(5000)
        else:
            raise FileNotFoundError(
                f"No station data found. Run station_spatial_analysis.py first."
            )

    radii_labels = [f"{r // 1000}km" if r >= 1000 else f"{r}m" for r in RADII]

    # ------------------------------------------------------------------
    # Step 2: Descriptive statistics
    # ------------------------------------------------------------------
    desc = print_descriptives(df, radii_labels)

    # ------------------------------------------------------------------
    # Step 3: Run regressions
    # ------------------------------------------------------------------
    print("\n" + "=" * 80)
    print("REGRESSION RESULTS")
    print("=" * 80)

    try:
        import statsmodels
        print("  Using statsmodels")
        results = run_regressions_statsmodels(df, radii_labels)
    except ImportError:
        print("  statsmodels not available. Using manual OLS fallback.")
        results = run_regressions_fallback(df, radii_labels)

    # Display results
    display_cols = ["radius", "model", "beta", "se", "p_value", "r_squared", "n_obs", "y_mean", "x_mean"]
    available_cols = [c for c in display_cols if c in results.columns]
    print("\n" + results[available_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}" if not pd.isna(x) else "—"))

    # ------------------------------------------------------------------
    # Step 4: Export
    # ------------------------------------------------------------------
    results.to_csv(REGRESSION_TABLE, index=False)
    print(f"\n  Results CSV: {REGRESSION_TABLE}")

    try:
        export_latex_table(results, REGRESSION_TEX)
    except Exception as e:
        print(f"  LaTeX export failed: {e}")

    # ------------------------------------------------------------------
    # Step 5: Interpretation summary
    # ------------------------------------------------------------------
    print("\n" + "=" * 80)
    print("INTERPRETATION GUIDE")
    print("=" * 80)
    print("""
    The coefficient β on has_accident represents the change in probability
    of having a union branch nearby, associated with having at least one
    accident nearby (within the specified radius).

    Key specifications to highlight in your paper:
    - LPM (binary): Most transparent. β directly interpretable as a
      probability difference. Use HC1 SEs since outcome is binary.
    - LPM + spatial: Controls for geographic location (latitude/longitude
      polynomial) to absorb spatial variation in both accidents and unions
      driven by station density, urbanization, etc.
    - Logit marginal effects: For robustness. Should give similar estimates
      to the LPM in well-specified models.

    IMPORTANT CAVEATS for identification:
    1. This is a CROSS-SECTIONAL correlation, not causal.
    2. Both accidents and unions are more likely near large/busy stations.
       Station size/traffic is an omitted variable.
    3. If you have the original HGISe shapefile with ST_TYPE (station type)
       and ST_OPEN (opening year), these would be valuable controls.
    4. Consider controlling for the train_operator from accident data
       (maps to railway company, a key confounder).
    5. The multi-radius results help assess sensitivity to the arbitrary
       5km buffer choice.
    """)


if __name__ == "__main__":
    main()

STATION-LEVEL REGRESSION: ACCIDENTS → UNION BRANCHES
Loading spatial data for multi-radius analysis...
  Stations: 2727, Accidents: 2038, Branches: 185
  Computing buffer radius = 1km...
    Stations with accident: 167 (6.1%)
    Stations with union:    124 (4.5%)
  Computing buffer radius = 2km...
    Stations with accident: 262 (9.6%)
    Stations with union:    205 (7.5%)
  Computing buffer radius = 5km...
    Stations with accident: 558 (20.5%)
    Stations with union:    473 (17.3%)
  Computing buffer radius = 10km...
    Stations with accident: 1074 (39.4%)
    Stations with union:    885 (32.5%)
  Computing buffer radius = 15km...
    Stations with accident: 1505 (55.2%)
    Stations with union:    1228 (45.0%)

  Multi-radius dataset saved: C:\Users\Keitaro Ninomiya\Box\Research Notes (keitaro2@illinois.edu)\RailwayUnions\Processed_Data\station_multi_radius.csv

DESCRIPTIVE STATISTICS
  Total stations: 2727
Radius Stations w/ accident Stations w/ union Mean acc. count Mean unio

In [ ]:
"""
Regression Results Viewer
=========================
Renders the regression table + coefficient plot in your browser.

Usage in VS Code:
    1. Run this script (F5 or terminal: python show_regression_results.py)
    2. It opens an interactive HTML page in your default browser
    3. Press Ctrl+C in terminal to stop the server

Alternatively, set SAVE_ONLY = True to just write the HTML file
and open it manually with VS Code's Live Preview extension.
"""

import webbrowser
import http.server
import threading
import os
import tempfile

# Set True to skip the server and just save the file
SAVE_ONLY = False

# Where to save
base_path = r"C:\Users\Keitaro Ninomiya\Box\Research Notes (keitaro2@illinois.edu)\RailwayUnions\Processed_Data"
OUTPUT_HTML = os.path.join(base_path, "regression_results_viewer.html")


# =========================================================================
# RESULTS DATA
# =========================================================================

RESULTS = {
    "1km": {
        "lpm":      {"b": 0.6851, "se": 0.0359, "p": 0.0000, "r2": 0.622},
        "lpm_sp":   {"b": 0.6825, "se": 0.0356, "p": 0.0000, "r2": 0.626},
        "count":    {"b": 0.0199, "se": 0.0030, "p": 0.0000, "r2": 0.271},
        "logit":    {"b": 0.0334, "se": 0.0087, "p": 0.0001, "r2": 0.676},
        "logit_sp": {"b": 0.0130, "se": 0.0056, "p": 0.0195, "r2": 0.728},
        "y_mean": 0.045, "x_mean": 0.061,
        "n_acc": 167, "n_uni": 124,
        "p_yes": 0.689, "p_no": 0.004,
    },
    "2km": {
        "lpm":      {"b": 0.7233, "se": 0.0275, "p": 0.0000, "r2": 0.654},
        "lpm_sp":   {"b": 0.7174, "se": 0.0273, "p": 0.0000, "r2": 0.660},
        "count":    {"b": 0.0134, "se": 0.0019, "p": 0.0000, "r2": 0.257},
        "logit":    {"b": 0.0622, "se": 0.0123, "p": 0.0000, "r2": 0.671},
        "logit_sp": {"b": 0.0188, "se": 0.0072, "p": 0.0089, "r2": 0.733},
        "y_mean": 0.075, "x_mean": 0.096,
        "n_acc": 262, "n_uni": 205,
        "p_yes": 0.729, "p_no": 0.006,
    },
    "5km": {
        "lpm":      {"b": 0.8049, "se": 0.0166, "p": 0.0000, "r2": 0.735},
        "lpm_sp":   {"b": 0.7899, "se": 0.0165, "p": 0.0000, "r2": 0.749},
        "count":    {"b": 0.0056, "se": 0.0005, "p": 0.0000, "r2": 0.201},
        "logit":    {"b": 0.1833, "se": 0.0257, "p": 0.0000, "r2": 0.700},
        "logit_sp": {"b": 0.0654, "se": 0.0170, "p": 0.0001, "r2": 0.787},
        "y_mean": 0.173, "x_mean": 0.205,
        "n_acc": 558, "n_uni": 473,
        "p_yes": 0.814, "p_no": 0.009,
    },
    "10km": {
        "lpm":      {"b": 0.8071, "se": 0.0121, "p": 0.0000, "r2": 0.710},
        "lpm_sp":   {"b": 0.7736, "se": 0.0119, "p": 0.0000, "r2": 0.752},
        "count":    {"b": 0.0030, "se": 0.0001, "p": 0.0000, "r2": 0.166},
        "logit":    {"b": 0.4725, "se": 0.0540, "p": 0.0000, "r2": 0.661},
        "logit_sp": {"b": 0.1939, "se": 0.0411, "p": 0.0000, "r2": 0.809},
        "y_mean": 0.325, "x_mean": 0.394,
        "n_acc": 1074, "n_uni": 885,
        "p_yes": 0.814, "p_no": 0.007,
    },
    "15km": {
        "lpm":      {"b": 0.8115, "se": 0.0101, "p": 0.0000, "r2": 0.658},
        "lpm_sp":   {"b": 0.7556, "se": 0.0104, "p": 0.0000, "r2": 0.745},
        "count":    {"b": 0.0022, "se": 0.0001, "p": 0.0000, "r2": 0.157},
        "logit":    {"b": 0.8612, "se": 0.1021, "p": 0.0000, "r2": 0.604},
        "logit_sp": {"b": 0.4498, "se": 0.0969, "p": 0.0000, "r2": 0.814},
        "y_mean": 0.450, "x_mean": 0.552,
        "n_acc": 1505, "n_uni": 1228,
        "p_yes": 0.814, "p_no": 0.002,
    },
}


# =========================================================================
# BUILD HTML
# =========================================================================

def stars(p):
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return ""

def stars_html(p):
    s = stars(p)
    return f'<span class="stars">{s}</span>' if s else ""

def coef_cell(d):
    return f'{d["b"]:.4f}{stars_html(d["p"])}'

def se_cell(d):
    return f'({d["se"]:.4f})'

def build_html():
    radii = ["1km", "2km", "5km", "10km", "15km"]
    R = RESULTS

    # --- Regression table rows ---
    def reg_row(label, model, cls="coef"):
        cells = "".join(f"<td>{coef_cell(R[r][model])}</td>" for r in radii)
        return f'<tr class="{cls}"><td>{label}</td>{cells}</tr>'

    def se_row(model):
        cells = "".join(f"<td>{se_cell(R[r][model])}</td>" for r in radii)
        return f'<tr class="se"><td></td>{cells}</tr>'

    def footer_row(label, vals):
        cells = "".join(f"<td>{v}</td>" for v in vals)
        return f'<tr class="footer"><td><em>{label}</em></td>{cells}</tr>'

    # --- Coefficient plot data as JSON ---
    import json
    plot_data = {}
    for model in ["lpm", "lpm_sp", "logit_sp"]:
        plot_data[model] = []
        for r in radii:
            d = R[r][model]
            plot_data[model].append({
                "radius": r,
                "b": d["b"],
                "lo": d["b"] - 1.96 * d["se"],
                "hi": d["b"] + 1.96 * d["se"],
            })
    plot_json = json.dumps(plot_data)

    # --- Descriptives table ---
    desc_rows = ""
    for r in radii:
        d = R[r]
        desc_rows += f"""<tr>
            <td>{r}</td>
            <td>{d['n_acc']} ({d['x_mean']:.1%})</td>
            <td>{d['n_uni']} ({d['y_mean']:.1%})</td>
            <td class="hl">{d['p_yes']:.3f}</td>
            <td>{d['p_no']:.3f}</td>
            <td class="hl">{d['p_yes'] - d['p_no']:.3f}</td>
        </tr>"""

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Regression Results: Accidents and Union Branches</title>
<style>
  @import url('https://fonts.googleapis.com/css2?family=Crimson+Pro:ital,wght@0,400;0,600;0,700;1,400&family=IBM+Plex+Mono:wght@400;500&display=swap');

  :root {{
    --bg: #faf9f7;
    --fg: #1a1a1a;
    --muted: #777;
    --border: #ccc;
    --accent: #8b2500;
    --blue: #2c5985;
    --blue-l: #5a9fd4;
    --hl: #fff8e1;
  }}
  * {{ margin:0; padding:0; box-sizing:border-box; }}
  body {{
    font-family: 'Crimson Pro', Georgia, serif;
    background: var(--bg); color: var(--fg);
    padding: 36px 44px; max-width: 1060px; margin: 0 auto;
    line-height: 1.5;
  }}
  h1 {{ font-size: 21px; font-weight: 700; letter-spacing: -0.3px; }}
  .sub {{ font-size: 13.5px; color: var(--muted); margin-bottom: 28px; }}

  /* Tabs */
  .tabs {{ display:flex; gap:0; border-bottom: 2px solid var(--fg); margin-bottom: 24px; }}
  .tab {{
    padding: 7px 18px; cursor: pointer;
    font-family: 'IBM Plex Mono', monospace; font-size: 11.5px;
    font-weight: 500; text-transform: uppercase; letter-spacing: 0.4px;
    border: 2px solid transparent; border-bottom: none;
    color: var(--muted); background: transparent;
    position: relative; bottom: -2px; transition: 0.12s;
  }}
  .tab:hover {{ color: var(--fg); }}
  .tab.active {{ color: var(--fg); background: var(--bg); border-color: var(--fg); border-bottom-color: var(--bg); }}
  .pnl {{ display:none; }} .pnl.active {{ display:block; }}

  /* Reg table */
  table.reg {{ width:100%; border-collapse:collapse; font-size:13px; }}
  table.reg th, table.reg td {{ padding: 6px 12px; text-align:center; vertical-align:top; }}
  table.reg th {{ font-family:'IBM Plex Mono',monospace; font-size:11px; font-weight:500; color:var(--muted); }}
  table.reg td:first-child, table.reg th:first-child {{ text-align:left; padding-left:0; }}
  .topline {{ border-top: 2.5px solid var(--fg); }}
  .midline td {{ border-top: 1px solid var(--border); }}
  .botline td {{ border-top: 2.5px solid var(--fg); }}
  .ph td {{ font-style:italic; color:var(--muted); padding-top:12px; padding-bottom:4px; font-size:12.5px; }}
  .coef td {{ font-family:'IBM Plex Mono',monospace; font-size:12px; }}
  .se td {{ font-family:'IBM Plex Mono',monospace; font-size:11px; color:var(--muted); padding-top:0; padding-bottom:8px; }}
  .footer td {{ font-size:11.5px; color:var(--muted); padding:2px 12px 2px 0; }}
  .stars {{ color: var(--accent); font-weight:600; }}
  .notes {{ font-size:11px; color:var(--muted); margin-top:10px; padding-top:10px; border-top:1px solid var(--border); line-height:1.5; }}

  /* Descriptives */
  table.desc {{ width:100%; border-collapse:collapse; font-size:12.5px; }}
  table.desc th {{
    font-family:'IBM Plex Mono',monospace; font-size:10.5px; font-weight:500;
    color:var(--muted); text-transform:uppercase; letter-spacing:0.3px;
    padding:7px 10px; text-align:center; border-bottom:2px solid var(--fg);
  }}
  table.desc th:first-child {{ text-align:left; padding-left:0; }}
  table.desc td {{ padding:6px 10px; text-align:center; font-family:'IBM Plex Mono',monospace; font-size:11.5px; border-bottom:1px solid #eee; }}
  table.desc td:first-child {{ text-align:left; padding-left:0; font-family:'Crimson Pro',serif; font-size:13px; }}
  table.desc tr:last-child td {{ border-bottom:2px solid var(--fg); }}
  .hl {{ background: var(--hl); font-weight:500; }}

  /* Coef plot */
  .plot-title {{ font-size:14px; font-weight:600; margin-bottom:2px; }}
  .plot-sub {{ font-size:12px; color:var(--muted); margin-bottom:16px; }}
  .mbtn-group {{ display:flex; gap:6px; margin-bottom:18px; flex-wrap:wrap; }}
  .mbtn {{
    padding:4px 13px; font-family:'IBM Plex Mono',monospace; font-size:10.5px;
    border:1.5px solid var(--border); background:#fff; cursor:pointer; transition:0.1s;
  }}
  .mbtn:hover {{ border-color:var(--fg); }}
  .mbtn.active {{ background:var(--fg); color:#fff; border-color:var(--fg); }}

  canvas {{ display:block; width:100%; max-width:700px; }}
</style>
</head>
<body>

<h1>Table 1: Accidents and Union Branch Establishment</h1>
<p class="sub">Dep. Var.: &#x1D7D9;(Union Branch within <em>r</em>&thinsp;km) &middot; 2,727 stations &middot; 1861 England, Wales &amp; Scotland</p>

<div class="tabs">
  <div class="tab active" onclick="showTab('reg')">Regression Table</div>
  <div class="tab" onclick="showTab('plot')">Coefficient Plot</div>
  <div class="tab" onclick="showTab('desc')">Descriptives</div>
</div>

<!-- ====== REGRESSION TABLE ====== -->
<div class="pnl active" id="pnl-reg">
<table class="reg">
<thead>
  <tr class="topline"><th style="min-width:240px"></th><th>(1)</th><th>(2)</th><th>(3)</th><th>(4)</th><th>(5)</th></tr>
  <tr><th></th><th>1 km</th><th>2 km</th><th>5 km</th><th>10 km</th><th>15 km</th></tr>
</thead>
<tbody>
  <tr class="ph midline"><td colspan="6">Panel A: Extensive Margin (OLS / LPM)</td></tr>
  {reg_row("has_accident", "lpm")}
  {se_row("lpm")}
  {reg_row('has_accident <span style="color:var(--muted);font-style:italic;font-size:11px">(+ spatial)</span>', "lpm_sp")}
  {se_row("lpm_sp")}

  <tr class="ph midline"><td colspan="6">Panel B: Intensive Margin (OLS / LPM)</td></tr>
  {reg_row("accident_count", "count")}
  {se_row("count")}

  <tr class="ph midline"><td colspan="6">Panel C: Logit (Marginal Effects at Mean)</td></tr>
  {reg_row("has_accident", "logit")}
  {se_row("logit")}
  {reg_row('has_accident <span style="color:var(--muted);font-style:italic;font-size:11px">(+ spatial)</span>', "logit_sp")}
  {se_row("logit_sp")}

  {footer_row("R&sup2; (A, no controls)", [f'{R[r]["lpm"]["r2"]:.3f}' for r in radii])}
  {footer_row("R&sup2; (A, + spatial)", [f'{R[r]["lpm_sp"]["r2"]:.3f}' for r in radii])}
  {footer_row("Pseudo-R&sup2; (C, + spatial)", [f'{R[r]["logit_sp"]["r2"]:.3f}' for r in radii])}
  {footer_row("Dep. var. mean", [f'{R[r]["y_mean"]:.3f}' for r in radii])}
  {footer_row("Treatment mean", [f'{R[r]["x_mean"]:.3f}' for r in radii])}
  {footer_row("Observations", ["2,727"]*5)}
  {footer_row("Spatial controls", ["rows 2, 4"]*5)}
</tbody>
</table>
<div class="notes">
  <em>Notes:</em> <span class="stars">***</span>&thinsp;p&lt;0.01, <span class="stars">**</span>&thinsp;p&lt;0.05, <span class="stars">*</span>&thinsp;p&lt;0.10.
  HC1 robust SEs in parentheses.
  Spatial controls: lat, lon, lat&sup2;, lon&sup2;, lat&times;lon.
  Unit: railway station (1861). Dep. var.: &ge;1 ASRS branch within <em>r</em>.
  Treatment: &ge;1 recorded accident within <em>r</em>.
</div>
</div>

<!-- ====== COEFFICIENT PLOT ====== -->
<div class="pnl" id="pnl-plot">
  <p class="plot-title">Coefficient on has_accident across buffer radii</p>
  <p class="plot-sub">Point estimates with 95% CI</p>
  <div class="mbtn-group">
    <div class="mbtn active" onclick="setModel('lpm',this)">LPM</div>
    <div class="mbtn" onclick="setModel('lpm_sp',this)">LPM + Spatial</div>
    <div class="mbtn" onclick="setModel('logit_sp',this)">Logit + Spatial (ME)</div>
  </div>
  <canvas id="cvs" height="260"></canvas>
</div>

<!-- ====== DESCRIPTIVES ====== -->
<div class="pnl" id="pnl-desc">
<h2 style="font-size:16px;margin-bottom:14px;">Descriptive Statistics by Buffer Radius</h2>
<table class="desc">
<thead>
  <tr>
    <th>Radius</th>
    <th>Stns w/ accident</th>
    <th>Stns w/ union</th>
    <th>P(union|acc)</th>
    <th>P(union|no acc)</th>
    <th>Raw diff.</th>
  </tr>
</thead>
<tbody>
  {desc_rows}
</tbody>
</table>
<div class="notes" style="margin-top:16px">
  <em>Notes:</em> "P(union|acc)" is the share of stations with &ge;1 accident that also have &ge;1 union branch.
  The raw difference is a simple difference-in-means, not regression-adjusted.
  N = 2,727 stations in all rows.
</div>
</div>

<script>
// Tab switching
function showTab(id) {{
  document.querySelectorAll('.pnl').forEach(p => p.classList.remove('active'));
  document.querySelectorAll('.tab').forEach(t => t.classList.remove('active'));
  document.getElementById('pnl-' + id).classList.add('active');
  event.target.classList.add('active');
  if (id === 'plot') drawPlot();
}}

// Coefficient plot
const DATA = {plot_json};
let currentModel = 'lpm';

function setModel(m, el) {{
  currentModel = m;
  document.querySelectorAll('.mbtn').forEach(b => b.classList.remove('active'));
  el.classList.add('active');
  drawPlot();
}}

function drawPlot() {{
  const canvas = document.getElementById('cvs');
  const ctx = canvas.getContext('2d');
  const dpr = window.devicePixelRatio || 1;

  // Size
  const W = canvas.parentElement.clientWidth;
  const H = 260;
  canvas.width = W * dpr;
  canvas.height = H * dpr;
  canvas.style.width = W + 'px';
  canvas.style.height = H + 'px';
  ctx.scale(dpr, dpr);

  ctx.clearRect(0, 0, W, H);

  const pts = DATA[currentModel];
  const pad = {{ l: 70, r: 30, t: 20, b: 40 }};
  const pw = W - pad.l - pad.r;
  const ph = H - pad.t - pad.b;

  // Determine scale
  let allVals = [];
  pts.forEach(p => {{ allVals.push(p.lo, p.hi, p.b); }});
  let mn = Math.min(...allVals, 0);
  let mx = Math.max(...allVals);
  const margin = (mx - mn) * 0.12;
  mn -= margin; mx += margin;

  function xPos(v) {{ return pad.l + ((v - mn) / (mx - mn)) * pw; }}
  function yPos(i) {{ return pad.t + i * (ph / (pts.length - 1 || 1)); }}

  // Zero line
  if (mn <= 0 && mx >= 0) {{
    const x0 = xPos(0);
    ctx.strokeStyle = '#ccc';
    ctx.lineWidth = 1;
    ctx.setLineDash([4, 3]);
    ctx.beginPath(); ctx.moveTo(x0, pad.t - 5); ctx.lineTo(x0, H - pad.b + 5); ctx.stroke();
    ctx.setLineDash([]);
  }}

  // Grid
  ctx.strokeStyle = '#eee';
  ctx.lineWidth = 0.5;
  const nTicks = 5;
  for (let i = 0; i <= nTicks; i++) {{
    const v = mn + (mx - mn) * i / nTicks;
    const x = xPos(v);
    ctx.beginPath(); ctx.moveTo(x, pad.t); ctx.lineTo(x, H - pad.b); ctx.stroke();
    ctx.fillStyle = '#999';
    ctx.font = '10px IBM Plex Mono, monospace';
    ctx.textAlign = 'center';
    ctx.fillText(v.toFixed(2), x, H - pad.b + 16);
  }}

  // Points
  pts.forEach((p, i) => {{
    const y = yPos(i);
    const xL = xPos(p.lo);
    const xH = xPos(p.hi);
    const xB = xPos(p.b);

    // CI line
    ctx.strokeStyle = '#5a9fd4';
    ctx.lineWidth = 2;
    ctx.beginPath(); ctx.moveTo(xL, y); ctx.lineTo(xH, y); ctx.stroke();

    // CI caps
    ctx.beginPath(); ctx.moveTo(xL, y - 5); ctx.lineTo(xL, y + 5); ctx.stroke();
    ctx.beginPath(); ctx.moveTo(xH, y - 5); ctx.lineTo(xH, y + 5); ctx.stroke();

    // Dot
    ctx.fillStyle = '#2c5985';
    ctx.beginPath(); ctx.arc(xB, y, 5, 0, Math.PI * 2); ctx.fill();
    ctx.strokeStyle = '#fff';
    ctx.lineWidth = 1.5;
    ctx.beginPath(); ctx.arc(xB, y, 5, 0, Math.PI * 2); ctx.stroke();

    // Label
    ctx.fillStyle = '#555';
    ctx.font = '12px IBM Plex Mono, monospace';
    ctx.textAlign = 'right';
    ctx.fillText(p.radius, pad.l - 12, y + 4);

    // Value
    ctx.textAlign = 'left';
    ctx.fillStyle = '#888';
    ctx.font = '10px IBM Plex Mono, monospace';
    ctx.fillText(p.b.toFixed(4), xH + 8, y + 3);
  }});
}}

// Initial draw if plot tab is visible
window.addEventListener('load', () => {{
  if (document.getElementById('pnl-plot').classList.contains('active')) drawPlot();
}});
window.addEventListener('resize', () => {{
  if (document.getElementById('pnl-plot').classList.contains('active')) drawPlot();
}});
</script>
</body>
</html>"""
    return html


# =========================================================================
# SERVE OR SAVE
# =========================================================================

def main():
    html = build_html()

    # Save the file
    with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"HTML saved to: {OUTPUT_HTML}")

    if SAVE_ONLY:
        print("Open this file in VS Code and use Live Preview (Ctrl+Shift+P → 'Live Preview')")
        print("Or open directly in browser.")
        return

    # Serve on localhost and open browser
    PORT = 8765

    class Handler(http.server.SimpleHTTPRequestHandler):
        def __init__(self, *args, **kwargs):
            super().__init__(*args, directory=os.path.dirname(OUTPUT_HTML), **kwargs)
        def log_message(self, format, *args):
            pass  # Suppress server logs

    server = http.server.HTTPServer(("localhost", PORT), Handler)
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()

    url = f"http://localhost:{PORT}/{os.path.basename(OUTPUT_HTML)}"
    print(f"Opening {url}")
    webbrowser.open(url)

    print("Press Ctrl+C to stop the server.")
    try:
        thread.join()
    except KeyboardInterrupt:
        print("\nServer stopped.")
        server.shutdown()


if __name__ == "__main__":
    main()

HTML saved to: C:\Users\Keitaro Ninomiya\Box\Research Notes (keitaro2@illinois.edu)\RailwayUnions\Processed_Data\regression_results_viewer.html
Opening http://localhost:8765/regression_results_viewer.html
Press Ctrl+C to stop the server.
